## Pattern support

Based on verb pattern tables in *vp_data3.db*, a new database with a new table *pattern_support* is created. This table shows verb (compound) frequencies, absolute support of full verb patterns and realtive support of full verb patterns among transactions.

The resulting table will be exported to *target_data* directory in CSV-format.

In [1]:
import sys

sys.path.append('../../../common_code')

In [2]:
import sqlite3
from db_operations.db_display import *

## Input parameters

In [3]:
RESULT_DB = "pattern_support.db"
VP_DATA_DB = "vp_data3.db"

## Data processing

In [8]:
con = sqlite3.connect(RESULT_DB)
cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{VP_DATA_DB}" AS vp')

cur.execute("""
DROP TABLE IF EXISTS pattern_support
""")

cur.execute("""
CREATE TABLE pattern_support AS
SELECT
    patterns_meta.pat_id,
    verb_word,
    verb_compound,
    phrase_case,
    adp,
    inf_verb,
    verb_match_count AS verb_occurrence_count,
    phrase_count AS absolute_support,
    CAST(phrase_count AS REAL) / CAST(verb_match_count AS REAL) * 100 AS relative_support
FROM
(
    SELECT 
        patterns.pat_id,
        verb_word,
        verb_compound,
        phrase_case,
        adp,
        inf_verb,
        count(*) AS verb_match_count
    FROM
        vp.verb_matches AS verb_matches
    INNER JOIN
        vp.patterns AS patterns
    ON
        patterns.pat_id = verb_matches.pat_id
    GROUP BY
        patterns.pat_id
) as tbl
INNER JOIN
    vp.patterns_meta AS patterns_meta
ON
    tbl.pat_id = patterns_meta.pat_id
ORDER BY
    relative_support DESC
""")

con.close()

## Result

In [6]:
display_sqlite_as_dataframe(RESULT_DB, 'pattern_support', 10)

,pat_id,verb_word,verb_compound,phrase_case,adp,inf_verb,verb_occurrence_count,absolute_support,relative_support
0,50,andma,andeks,all,,,7,7,100.000000
1,52,paluma,andeks,abl,,,3,3,100.000000
2,214,luiskama,ette,all,,,1,1,100.000000
3,874,mängima,kätte,all,,,4,4,100.000000
4,927,harjutama,külge,all,,,1,1,100.000000
5,1010,liidendama,,kom,,,1,1,100.000000
6,1648,sarnlema,,all,,,2,2,100.000000
7,2042,tegema,tuupi,all,,,10,10,100.000000
8,1813,tegema,säru,all,,,38,37,97.368421
9,2054,valguma,täis,part,,,63,61,96.825397
